# Preprocessing

Before training the predictive models, a preprocessing pipeline was applied to prepare the dataset and improve data quality. This step included handling missing values, transforming categorical variables, feature engineering, and removing unnecessary attributes.

Given the clinical nature of the MIMIC-III dataset, different strategies were used to handle missing values depending on the type of variable. Continuous physiological and laboratory measurements were mainly imputed using the median, while intervention- and medication-related variables were filled with zero when appropriate, since missing values in these cases often indicate that a procedure or medication was not administered.

Categorical variables were transformed using one-hot encoding. For high-cardinality variables such as diagnosis, only the most frequent categories were preserved, while less frequent diagnoses were grouped into a common “OTHER” category to reduce dimensionality and sparsity.

Additionally, some features were engineered during preprocessing. For example, the date of birth was converted into patient age, which is more clinically relevant for predictive modeling. Identifier columns and non-informative attributes were removed before model training.

In [40]:
import pandas as pd
from src.feature_groups_map import feature_groups
from IPython.display import display

In [41]:
df = pd.read_csv("data/datasets/final_selected.csv")
diagnosis_map_df = pd.read_csv("data/datasets/diagnosis_map.csv")

In [42]:
df.shape

(62722, 159)

We removed patients who died during the ICU stay. These cases may introduce strong outcome-related patterns that are not aligned with the main prediction objective and could negatively affect model training. After filtering, the EXPIRE_FLAG column is removed since it is no longer needed.

In [43]:
df = df[df["EXPIRE_FLAG"] == 1].drop(columns=["EXPIRE_FLAG"])

In [44]:
def get_missing_info(df):
    missing_info = pd.DataFrame({
        "null_count": df.isnull().sum(),
        "null_percentage": (df.isnull().sum() / len(df)) * 100
    })

    missing_info = missing_info.sort_values(
        by="null_percentage",
        ascending=False
    )

    display(
    missing_info.style.set_table_attributes(
        'style="display:inline-block; max-height:500px; overflow:auto;"'
    )
)

    return missing_info

In [45]:
missing_info = get_missing_info(df)

,null_count,null_percentage
dexmedetomidine_sum,24684,99.793814
dobutamine_sum,24653,99.668486
dobutamine_count,24653,99.668486
milrinone_sum,24625,99.555286
milrinone_count,24625,99.555286
cisatracurium_sum,24599,99.450172
cisatracurium_count,24599,99.450172
cryoprecipitate_sum,24589,99.409743
tpn_sum,24587,99.401658
tpn_count,24587,99.401658


Since the target variable of this project is Length of Stay (LOS), all rows with missing LOS values were removed from the dataset, as these samples cannot be used for supervised learning. Additionally, rows with missing diagnosis information were also excluded. As only a small number of records (approximately 17) lacked diagnosis data, removing them had minimal impact on the overall dataset while helping maintain data consistency during preprocessing.

In [47]:
df = df[df["LOS"].notnull()]
df = df[df["DIAGNOSIS"].notnull()]

To handle missing values, different imputation strategies were applied according to the semantic meaning of each feature group. Variables related to medications, interventions, procedures, outputs, and device usage were imputed with zero, since missing values in these cases often indicate that the event or intervention did not occur during the patient stay. On the other hand, continuous physiological and laboratory measurements were imputed using the median value of each feature. Median imputation was chosen because it is more robust to outliers and skewed distributions, which are common in clinical datasets such as MIMIC-III.

In [48]:
for col in feature_groups["zero_impute"]:
    if col in df.columns:
        df[col] = df[col].fillna(0)

In [49]:
for col in feature_groups["median_impute"]:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

In [50]:
missing_info = get_missing_info(df)

,null_count,null_percentage
gcs_verbal_avg,0,0.000000
gcs_eye_avg,0,0.000000
gcs_verbal_min,0,0.000000
systolic_bp_min,0,0.000000
respiratory_rate_min,0,0.000000
ph_blood_latest,0,0.000000
gcs_eye_min,0,0.000000
gcs_total_avg,0,0.000000
gcs_motor_avg,0,0.000000
gcs_verbal_max,0,0.000000


In [51]:
categorical_columns = df.select_dtypes(
    include=["object", "category", "bool"]
).columns

print(categorical_columns)

Index(['ADMISSION_TYPE', 'DIAGNOSIS', 'DOB', 'ADMITTIME'], dtype='str')


C:\Users\belac\AppData\Local\Temp\ipykernel_18184\2331803844.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


After handling missing values, additional preprocessing steps were applied to prepare the categorical and temporal information for modeling. Since raw date variables are not directly suitable for machine learning algorithms, the patient’s date of birth was transformed into an age feature by calculating the difference between admission time and birth date. This approach provides a more clinically meaningful representation of patient demographics. After generating the age variable, the original DOB and ADMITTIME columns were removed from the dataset to avoid redundancy and reduce unnecessary temporal information during training.

In [52]:
df = df.copy()

df = df.assign(
    age=((pd.to_datetime(df["ADMITTIME"]) -
          pd.to_datetime(df["DOB"])).dt.days / 365.25).astype(int)
)

columns_to_drop = [
    "DOB",
    "ADMITTIME"
]

df = df.drop(columns=columns_to_drop)

In [53]:
len(df["DIAGNOSIS"].value_counts())

7641

The DIAGNOSIS column originally contained 15,248 unique categories, making it impractical to directly apply one-hot encoding or use the raw values for model training due to the extremely high dimensionality and sparsity that would be introduced into the dataset. To address this issue, a custom script named src.create_diagnosis_map was developed to automatically group diagnoses into a smaller set of clinically meaningful categories using a Large Language Model (LLM).

The diagnoses were classified into predefined medical groups, including categories such as cardiovascular, respiratory, infectious, neurological, gastrointestinal, renal, oncology, trauma-related conditions, among others. To improve processing efficiency, the classification pipeline was executed in parallel using 10 workers (MAX_WORKERS = 10), with batches of 50 diagnoses sent per API request (BATCH_SIZE = 50). Additionally, intermediate results were periodically saved every 10 completed batches (SAVE_EVERY_BATCHES = 10) to avoid losing progress during long-running executions.

After generating the diagnosis-category mapping, the original diagnosis values were replaced by their corresponding grouped category using a dictionary-based mapping approach. This significantly reduced the cardinality of the feature while preserving clinically relevant information for the predictive modeling task.

In [54]:
diagnosis_dict = dict(
    zip(
        diagnosis_map_df["diagnosis"],
        diagnosis_map_df["category"]
    )
)

df["DIAGNOSIS"] = df["DIAGNOSIS"].map(diagnosis_dict)

In [55]:
len(df["DIAGNOSIS"].value_counts())

20

In [56]:
df["DIAGNOSIS"].value_counts()

DIAGNOSIS
cardiovascular            5860
infectious                3294
neurological              2794
gastrointestinal          2279
symptoms_unspecified      2238
respiratory               1615
oncology                  1252
trauma_injury             1090
hepatobiliary              693
renal                      606
hematologic                524
endocrine_metabolic        502
other                      348
genitourinary              212
postoperative_surgical     138
musculoskeletal            136
toxicology_poisoning        88
psychiatric                 54
dermatologic                23
pregnancy_obstetric         16
Name: count, dtype: int64

After that, one-hot encoding was applied to transform categorical variables into a numerical representation suitable for machine learning models. This technique converts each categorical value into a binary feature, allowing the algorithms to interpret categorical information without introducing artificial ordinal relationships between categories.

In [57]:
df = pd.get_dummies(
    df,
    columns=["DIAGNOSIS", "ADMISSION_TYPE"],
    drop_first=True,
    dtype=int
)

In [58]:
df.head().shape

(5, 177)

In [59]:
df.head()

,gcs_verbal_avg,gcs_eye_avg,gcs_verbal_min,systolic_bp_min,respiratory_rate_min,ph_blood_latest,gcs_eye_min,gcs_total_avg,gcs_motor_avg,gcs_verbal_max,...,DIAGNOSIS_pregnancy_obstetric,DIAGNOSIS_psychiatric,DIAGNOSIS_renal,DIAGNOSIS_respiratory,DIAGNOSIS_symptoms_unspecified,DIAGNOSIS_toxicology_poisoning,DIAGNOSIS_trauma_injury,ADMISSION_TYPE_EMERGENCY,ADMISSION_TYPE_NEWBORN,ADMISSION_TYPE_URGENT
0,5.00,3.833333,5.0,89.0,15.0,7.40,3.0,13.571429,6.0,5.0,...,0,0,0,0,0,0,0,1,0,0
3,4.75,4.000000,4.0,95.0,11.0,118.00,4.0,13.571429,6.0,5.0,...,0,0,0,0,0,0,0,1,0,0
4,5.00,4.000000,5.0,120.0,16.0,16.00,4.0,15.000000,6.0,5.0,...,0,0,0,0,0,0,0,1,0,0
6,1.00,2.900000,1.0,31.0,0.0,7.39,1.0,9.700000,5.8,1.0,...,0,0,0,0,0,0,0,1,0,0
7,4.00,3.500000,4.0,120.0,11.0,3.70,3.0,13.571429,6.0,4.0,...,0,0,0,0,0,0,0,1,0,0


In [60]:
df.to_csv("data/datasets/final_processed.csv")